# Colab 10 — Systematic C1+C2 Discovery V1 (2,520 hypotheses)

**PUBLIC CODE / PRIVATE DATA.** This runs ONLY on an existing, owner-prepared MICRO ZIP. No Drive mount or original source reads. July 2024 is repeatedly inspected Development, NOT independent validation; August 2024 requires a separate, explicit owner grant before preparing a new ZIP in Colab 04. Do not use Validation 2025 or OOS 2026.

Run once in a **fresh** offline Colab runtime. **No fishing for best-looking hypotheses**: fixed 12 features, fixed quantile bins trained without outcomes on July 1–21, fixed 630 risk-set rules and 4 horizons, joint maxT over all 2,520. July report retains every null and negative hypothesis. If there is no qualifying candidate, **do not open August under this protocol**.

Historical executed-trade prices, costs and future returns are research **proxies**, never fill evidence, profit guarantee or live deployment. Save July plan, complete report and frozen replay manifest on your computer, not public GitHub.

In [ ]:
# 1. Clean no-Drive Colab, pinned public CODE from GitHub main.
import os,sys,subprocess,hashlib,json
from pathlib import Path
if any(Path(p).exists() for p in ('/content/drive/MyDrive','/content/gdrive/MyDrive','/gdrive/MyDrive')):
    raise RuntimeError('Drive mounted: disconnect/delete runtime and use offline Colab')
if any(os.environ.get(k) for k in ('GOOGLE_APPLICATION_CREDENTIALS','GOOGLE_OAUTH_ACCESS_TOKEN','GOOGLE_DRIVE_TOKEN')):
    raise RuntimeError('Credential environment detected; refuse')
repo=Path('/content/hypolab-systematic-discovery-public-code')
if repo.exists():raise RuntimeError('Fresh runtime required; old checkout found')
subprocess.run(['git','clone','--depth','1','https://github.com/DuStER1921/CryptoHypothesisLab.git',str(repo)],check=True,stdout=subprocess.DEVNULL)
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
sys.path.insert(0,str(repo/'src'))
from google.colab import files
from hypolab.discovery_v1 import (SCHEMA, JULY, AUGUST, make_discovery_plan,run_july_discovery,
    make_replay_manifest,run_august_replay,freeze_sha256,_canonical,_parse_plan)
from hypolab.micro_bundle import MAX_ARCHIVE_BYTES,MicroBundleError
print('PUBLIC CODE COMMIT:',commit)
print('DISCOVERY V1 FREEZE:',freeze_sha256())
print('No Google Drive mounted; original source unavailable')

In [ ]:
# 2. Exactly one analysis mode; no source read happens automatically.
mode=input('Mode? JULY (discover) or AUGUST (replay frozen candidates only): ').strip().upper()
if mode not in ('JULY','AUGUST'):raise RuntimeError('Mode must be JULY or AUGUST')
private=Path('/content/hypolab-systematic-discovery-private')
private.mkdir(mode=0o700,exist_ok=False)
print('Private local runtime:',private,'MODE:',mode)

In [ ]:
# 3. In AUGUST mode, upload July's saved plan/report/manifest FIRST.
#     Enforce empty shortlist refusal BEFORE requesting any August market ZIP.
if mode=='AUGUST':
    uploaded=files.upload()
    expected_names={'discovery-v1-feature-plan.json','discovery-v1-july-full-report.json','discovery-v1-august-frozen-manifest.json'}
    if set(uploaded)!=expected_names:raise RuntimeError('Three exact saved July artifacts required')
    plan=uploaded['discovery-v1-feature-plan.json']
    july_result_bytes=uploaded['discovery-v1-july-full-report.json']
    manifest=uploaded['discovery-v1-august-frozen-manifest.json']
    _parse_plan(plan)
    july_report=json.loads(july_result_bytes)
    if _canonical(july_report)!=july_result_bytes:raise RuntimeError('July report bytes not canonical')
    if make_replay_manifest(july_report,plan)!=manifest:raise RuntimeError('July plan/report/manifest chain FAILED')
    chosen=json.loads(manifest)['preselected_candidates']
    if not chosen:raise RuntimeError('NO QUALIFIED JULY CANDIDATES. AUGUST MUST NOT BE OPENED.')
    print('FROZEN CANDIDATES:',chosen)
    print('PLAN SHA:',hashlib.sha256(plan).hexdigest())
    print('MANIFEST SHA:',hashlib.sha256(manifest).hexdigest())
    del uploaded
else:
    print('JULY mode: no previous artifact required')

In [ ]:
# 4. Upload ONE authorized private MICRO ZIP and verify its SHA before analysis.
uploaded=files.upload()
if len(uploaded)!=1:raise RuntimeError('Upload exactly one MICRO ZIP')
name,raw=next(iter(uploaded.items()))
if not name.startswith('HYPOLAB_MICRO_DEV_') or not name.endswith('.zip') or not 0<len(raw)<=MAX_ARCHIVE_BYTES:
    raise RuntimeError('Invalid or oversized ZIP')
expected_sha=input('Exact SHA-256 of THIS MICRO ZIP: ').strip().lower()
actual_sha=hashlib.sha256(raw).hexdigest()
if len(expected_sha)!=64 or expected_sha!=actual_sha:raise RuntimeError('MICRO ZIP SHA MISMATCH')
if mode=='JULY' and not name.startswith('HYPOLAB_MICRO_DEV_2024-07_'):
    raise RuntimeError('JULY mode accepts July MICRO only')
if mode=='AUGUST' and not name.startswith('HYPOLAB_MICRO_DEV_2024-08_'):
    raise RuntimeError('AUGUST mode accepts August MICRO only')
packet=private/'private-verified-micro.zip'
with packet.open('xb') as fh:
    os.chmod(packet,0o600)
    fh.write(raw)
del raw,uploaded
print('Verified package SHA:',actual_sha)
if mode=='JULY':
    plan=make_discovery_plan(packet) # feature-only, no outcome labels
    print('TRAIN-ONLY FEATURE PLAN SHA:',hashlib.sha256(plan).hexdigest())
    print('FREEZE TOKEN:',json.loads(plan)['frozen_code_token'])
    plan_file=private/'discovery-v1-feature-plan.json'
    with plan_file.open('xb') as fh:fh.write(plan)
    print('Download and KEEP PLAN before any outcome computation.')
    files.download(str(plan_file))
else:
    print('August: no fit of thresholds, no new hypothesis search')

In [ ]:
# 5. JULY: test entire 2,520-member family; NEVER reselect by hand.
if mode=='JULY':
    report=run_july_discovery(packet,plan)
    raw_report=_canonical(report)
    report_file=private/'discovery-v1-july-full-report.json'
    with report_file.open('xb') as fh:fh.write(raw_report)
    manifest=make_replay_manifest(report,plan)
    manifest_file=private/'discovery-v1-august-frozen-manifest.json'
    with manifest_file.open('xb') as fh:fh.write(manifest)
    print('ALL hypotheses:',report['counts']['test_family_total'])
    print('With adequate support:',report['counts']['tested_with_support'])
    print('Fixed shortlisted candidates:',report['fixed_shortlist_for_next_development'])
    print('Complete private report SHA:',hashlib.sha256(raw_report).hexdigest())
    print('Frozen replay manifest SHA:',hashlib.sha256(manifest).hexdigest())
    print('ALL outcomes exploratory; not executable alpha')
    files.download(str(report_file))
    files.download(str(manifest_file))
    if not report['fixed_shortlist_for_next_development']:
        print('NO CANDIDATES: no August replay under this frozen protocol.')
else:
    print('JULY research not run in AUGUST replay session.')

In [ ]:
# 6. AUGUST: frozen one-shot replay of candidates only; no scanning/refitting.
if mode=='AUGUST':
    result=run_august_replay(packet,plan,manifest,july_result_bytes)
    report_path=private/'discovery-v1-august-replay-report.json'
    content=_canonical(result)
    with report_path.open('xb') as fh:fh.write(content)
    print('FROZEN AUGUST CANDIDATES:',result['candidate_results'])
    print('Private report SHA:',hashlib.sha256(content).hexdigest())
    print('No real order fills, quotes, live P&L or independent Validation/OOS')
    files.download(str(report_path))
else:
    print('No August data requested in JULY discovery session.')